# Polaris API traffic, v1 -- every operation, every reachable status. TRAFFIC ONLY.

**What this does.** Drives the same 286-cell grid as `polaris_log_coverage_v2.ipynb` -- the
same identities, fixture, doomed family, phases B-I and `Polaris-Request-Id`s -- and **stops
there**. It records what was asked of Polaris and what
Polaris answered.

**What it does not do.** It verifies nothing. No OpenSearch, no VictoriaLogs, no `kubectl`, no
Lua unit test, no gates, no invariants, no findings. That is `local-k8s`'s job
(`PLAN-split-traffic-and-verification.md`); the last section prints the run id, each phase's
start/end time and the request ids that job needs.

**Import boundary.** Same rule as `src/make_traffic.py`: nothing here imports `os_report`,
`log_coverage` or `vlogs`. The traffic helpers come from `traffic_helpers` (which
`log_coverage` only re-exports -- the same functions).

`local` env only, and it **mutates**: a catalog, principals, roles, namespaces, a table and a
view are created and deleted, and 500s are provoked. ~300 calls.

**No window waits.** v2 slept to a report-window boundary before every phase so each phase had a
window to itself. This notebook drives every phase back to back. Phases are recorded by wall-clock
start/end and every call carries its request id, so a verifier matches calls by id, not by window.

## 0 -- Setup

Polaris only. The run stops here if there is no root token.

In [ ]:
import pathlib
import sys
import time
from datetime import datetime, timezone

_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from polaris_test_utils import *  # noqa: F401,F403
import polaris_test_utils as ptu
import api_status_matrix as mx
import traffic_helpers as th     # the traffic half of log_coverage, same functions

init_env("local")
require_not_prod("the api traffic drive creates and deletes catalogs, principals, roles, "
                 "namespaces, a table and a view, and provokes 500s")

RUN = str(int(time.time()))


def _iso(epoch):
    """UTC ISO-8601 with milliseconds -- how every phase records when it ran."""
    return datetime.fromtimestamp(epoch, tz=timezone.utc).isoformat(timespec="milliseconds")


STARTED = datetime.now(timezone.utc)

try:
    ROOT_TOK = root_token()
except Exception as exc:
    raise RuntimeError(f"SETUP FAILED -- Polaris at {ptu.POLARIS_URL}: "
                       f"{type(exc).__name__}: {exc}") from exc

print(f"  polaris  OK   {ptu.POLARIS_URL}  realm={ptu.REALM}")
print(f"  run={RUN}  started={STARTED.isoformat()}")

## 1 -- Identities

Four: admin (root), the runner, a denied principal with a role and no grants, and a garbage
bearer. The runner is elected in section 2, once the fixture exists to grant it rights on.

In [ ]:
from polaris_rest import PolarisREST

adm_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)

RUNNER_NAME, RUNNER_ROLE = f"mx_{RUN}_runner", f"mx_{RUN}_runner_role"
DENIED_NAME, DENIED_ROLE = f"mx_{RUN}_denied", f"mx_{RUN}_denied_role"

TOKENS, IDENTITY_NOTES = {mx.ADMIN: ROOT_TOK}, []

# THE RUNNER IS ELECTED IN THE FIXTURE CELL, NOT HERE, and this is the whole
# lesson of run 1789007773. `provision_run_principal` grants the probe
# catalog's rights only when it is handed the fixture -- its own docstring says
# so: "catalog operations authorize against grant_records for the TARGET
# catalog and service_admin does not cover them". Provisioned here, before the
# fixture exists, the runner holds a role and no privileges, and 19 of the 2xx
# cells came back 403. The matrix then reported a Polaris that refuses every
# catalog operation, which is a statement about this cell.

_cid, _sec = th.provision_run_principal(adm_pc, DENIED_NAME, DENIED_ROLE, fixture=None,
                                        service_admin=False)
_r = adm_pc.get_token(_cid, _sec, scope=f"PRINCIPAL_ROLE:{DENIED_ROLE}")
if _r.status_code == 200:
    TOKENS[mx.DENIED] = _r.json()["access_token"]
    IDENTITY_NOTES.append(f"denied  {DENIED_NAME} (a role, and no grants at all)")
else:
    TOKENS[mx.DENIED] = mx.GARBAGE_TOKEN
    IDENTITY_NOTES.append(f"denied  NO TOKEN ({_r.status_code}) -- every 403 cell will report "
                          "401 instead and be MISSED, correctly")

for n in IDENTITY_NOTES:
    print(f"  {n}")
print("  nobody  a syntactically valid bearer no realm accepts")

## 2 -- The fixture, the binding, and the doomed family

One catalog, namespace, table and view. `BINDING` turns a spec template into a URL. Every
destructive grid operation is pointed at the `doomed_` family so the run does not delete its own
fixture. `teardown_all` is registered here, with atexit, so an abort anywhere below still cleans up.

In [ ]:
import api_surface as surf
from iceberg_rest import IcebergREST, build_create_table_payload, build_schema

adm_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)
run_adm_ic = adm_ic  # the doomed family is created as root; the grid deletes it as itself
SCHEMA = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

FX = surf.ProbeFixture(prefix=f"apimatrix{RUN}")
surf.setup_fixture(FX, adm_pc, adm_ic, ptu.BUCKET, ptu.MINIO_ENDPOINT, SCHEMA,
                   build_create_table_payload)
print(f"  fixture  {FX.cat} / {FX.ns} / {FX.tbl} / {FX.view}")

# NOW the runner can be elected: elect_drive_identity provisions it WITH the
# fixture (which is what grants the catalog rights) and then VERIFIES it with
# one management read and one catalog read, demoting to root loudly if it
# cannot do both. Run 1789007773 skipped both halves and spent 286 calls
# measuring a principal that could not read a table.
IDENTITY, run_pc, run_ic_elected, DRIVE_PRINCIPAL, IDENTITY_NOTES = th.elect_drive_identity(
    adm_pc, adm_ic, ptu.POLARIS_URL, ptu.REALM, RUNNER_NAME, RUNNER_ROLE, fixture=FX,
    root_token=ROOT_TOK)
TOKENS[mx.RUNNER] = run_pc.token or ROOT_TOK
for _n in IDENTITY_NOTES:
    print(f"  identity {_n}")
print(f"  runner   {DRIVE_PRINCIPAL} (elected: {IDENTITY})")
if DRIVE_PRINCIPAL == "root" or TOKENS[mx.RUNNER] == ROOT_TOK:
    print("  !! THE RUN IS DRIVING AS ROOT. That is a valid fallback and it VOIDS the")
    print("     per-principal margin: sum(principal.requests) == access_seen - parse_errors")
    print("     is then satisfied by a global counter with one row to sum, and proves nothing.")

BINDING = {
    "catalog": FX.cat, "namespace": FX.ns, "table": FX.tbl, "view": FX.view,
    "principalName": RUNNER_NAME, "principalRoleName": RUNNER_ROLE,
    "catalogRoleName": f"mx_{RUN}_crole", "plan-id": "no-such-plan-id",
    # names the grid CREATES rather than reads
    "new_catalog": f"mx{RUN}cat2", "new_namespace": f"mx_{RUN}_ns2",
    "new_table": f"mx_{RUN}_tbl2", "new_view": f"mx_{RUN}_vw2",
    "new_principal": f"mx_{RUN}_p2", "new_principal_role": f"mx_{RUN}_prole2",
    "new_catalog_role": f"mx_{RUN}_crole2",
    "renamed_table": f"{FX.tbl}_renamed", "renamed_view": f"{FX.view}_renamed",
    # THE DOOMED FAMILY. Every destructive operation in the grid is pointed
    # here by api_status_matrix.REBIND, and nothing else in the run reads them.
    "doomed_catalog": f"mx{RUN}doomed", "doomed_namespace": f"mx_{RUN}_ns_doomed",
    "doomed_table": f"mx_{RUN}_tbl_doomed", "doomed_view": f"mx_{RUN}_vw_doomed",
    "doomed_principal": f"mx_{RUN}_p_doomed",
    "doomed_principal_role": f"mx_{RUN}_prole_doomed",
    "doomed_catalog_role": f"mx_{RUN}_crole_doomed",
    "entity_version": 1, "run": RUN,
    "base_location": f"s3a://{ptu.BUCKET}/mx{RUN}cat2/",
    "allowed_location": f"s3a://{ptu.BUCKET}/",
    "s3_endpoint": ptu.MINIO_ENDPOINT, "s3_endpoint_internal": ptu.MINIO_ENDPOINT_INTERNAL,
    "metadata_location": f"s3a://{ptu.BUCKET}/mx{RUN}cat2/never-registered/metadata.json",
    "client_id": ptu.ROOT_CLIENT, "client_secret": ptu.ROOT_SECRET,
}

# The catalog role the role-graph cells READ has to exist before they read it.
_r = adm_pc.create_catalog_role(FX.cat, BINDING["catalogRoleName"])
print(f"  catalog role {BINDING['catalogRoleName']}: {_r.status_code}")

# THE DOOMED FAMILY. The happy sweep drives deleteCatalog, dropTable, dropView,
# dropNamespace, deletePrincipal and the two revokes -- bound naively they take
# the FIXTURE, and the run destroys itself at about call 30 of 63 with every
# later cell 404ing. api_status_matrix.REBIND points them here instead, so the
# deletes are real deletes of real entities and nothing the run still needs.
SETUP = {}
SETUP["doomed catalog"] = adm_pc.create_catalog(
    BINDING["doomed_catalog"], ptu.BUCKET, ptu.MINIO_ENDPOINT,
    minio_endpoint_internal=ptu.MINIO_ENDPOINT_INTERNAL,
).status_code
SETUP["doomed namespace"] = run_adm_ic.create_namespace(
    FX.cat, BINDING["doomed_namespace"]).status_code
SETUP["doomed table"] = run_adm_ic.create_table(
    FX.cat, FX.ns, build_create_table_payload(BINDING["doomed_table"], SCHEMA)).status_code
SETUP["doomed view"] = run_adm_ic.create_view(FX.cat, FX.ns, {
    "name": BINDING["doomed_view"], "schema": SCHEMA,
    "view-version": {"version-id": 1, "timestamp-ms": 0, "schema-id": 0,
                     "summary": {"engine-name": "api-status-matrix"},
                     "default-namespace": [FX.ns],
                     "representations": [{"type": "sql", "sql": "SELECT 1",
                                          "dialect": "spark"}]},
    "properties": {"mx.run": RUN}}).status_code
SETUP["doomed principal"] = adm_pc.create_principal(BINDING["doomed_principal"]).status_code
SETUP["doomed principal role"] = adm_pc.create_principal_role(
    BINDING["doomed_principal_role"]).status_code
SETUP["doomed catalog role"] = adm_pc.create_catalog_role(
    FX.cat, BINDING["doomed_catalog_role"]).status_code
# the two revokes need something assigned to revoke
SETUP["assign prole->principal"] = adm_pc.assign_principal_role_to_principal(
    BINDING["doomed_principal"], BINDING["doomed_principal_role"]).status_code
# ARGUMENT ORDER IS (catalog, principal_role, catalog_role) -- and this PUT can
# 404 immediately after create_catalog_role on PG-HA read-after-write lag, which
# is a known signature here, not a finding. Retry rather than record a false one.
SETUP["assign crole->prole"] = None
for _try in range(4):
    _ar = adm_pc.assign_catalog_role_to_principal_role(
        FX.cat, BINDING["doomed_principal_role"], BINDING["doomed_catalog_role"])
    SETUP["assign crole->prole"] = _ar.status_code
    if _ar.status_code != 404:
        break
    time.sleep(0.75)

for k, v in SETUP.items():
    print(f"  {k:26} {v}")
_bad = {k: v for k, v in SETUP.items() if not (200 <= int(v) < 300)}
if _bad:
    print(f"  !! setup did not fully succeed: {_bad}")
    print("     the delete cells for those entities will report 404 and be MISSED --")
    print("     correctly, and as a harness gap rather than a pipeline finding.")

# TEARDOWN THAT SURVIVES AN ABORT.
#
# Cleanup is section 11, eighteen cells below, and until now it was reachable
# only by running every cell between here and there. Any raise in between -- a
# gate, a transport error, an interrupted kernel -- left the catalog, the
# namespace, the table, the view, both principals and the whole doomed family
# on the cluster. That is where `apimatrix1789031469_cat` and `probe_ns` came
# from, and a leaked catalog is not tidiness: the next run's fixture collides
# with it and the collision is reported as a finding about Polaris.
#
# `teardown_all` is idempotent, never raises, and reads its clients out of
# globals(), so it is correct whether it runs from section 11, from atexit on
# kernel shutdown, or by hand in a fresh cell after a failure. Deletion order
# is children first: views and tables, then namespaces, then catalog roles,
# then catalogs. A 404 here is the expected answer for anything the grid
# already deleted as part of the test, and it is printed rather than hidden.
import atexit

_TORN = {"done": False}


def teardown_all(quiet=False):
    """Delete everything this run created. Idempotent, best-effort, never raises."""
    if _TORN["done"]:
        if not quiet:
            print("  teardown already ran -- nothing to do")
        return {}
    _TORN["done"] = True
    g = globals()
    ic = g.get("run_ic") or g.get("adm_ic")
    pc = g.get("adm_pc")
    out = {}

    def _try(label, fn):
        try:
            out[label] = fn().status_code
        except Exception as exc:
            out[label] = f"{type(exc).__name__}: {exc}"
        if not quiet:
            print(f"  {label:34} {out[label]}")

    if ic is not None:
        # THE RENAME DESTINATIONS, and leaving them out is what kept
        # `apimatrix1789365275_cat` on the cluster: renameTable/renameView
        # move `new_table`/`new_view` onto these names, nothing drops them,
        # so probe_ns answered 409 and the catalog delete 400. A teardown
        # that does not know every name the RUN can create is not a teardown.
        for _v in (FX.view, BINDING["doomed_view"], BINDING["new_view"],
                   BINDING["renamed_view"]):
            _try(f"drop view {_v}", lambda v=_v: ic.drop_view(FX.cat, FX.ns, v))
        for _t in (FX.tbl, BINDING["doomed_table"], BINDING["new_table"],
                   BINDING["renamed_table"]):
            _try(f"drop table {_t}", lambda t=_t: ic.drop_table(FX.cat, FX.ns, t))
        for _n in (BINDING["doomed_namespace"], BINDING["new_namespace"], FX.ns):
            _try(f"drop namespace {_n}", lambda n=_n: ic.drop_namespace(FX.cat, n))

    # THE SWEEP. Deleting a fixed list of names cannot be complete, because the
    # RUN creates entities it did not intend to: run 1789366561 left
    # `apimatrix..._cat` undeletable (400) after every named entity was gone,
    # and the likeliest reason is in its own findings -- `createNamespace`
    # answered 500 and COMMITTED anyway. A teardown that only knows the names it
    # planned will always trail the run by whatever the run did by accident.
    if ic is not None:
        try:
            _ns_resp = ic.list_namespaces(FX.cat)
            _nss = [n for n in (_ns_resp.json().get("namespaces") or [])] if _ns_resp.ok else []
        except Exception as exc:
            _nss = []
            if not quiet:
                print(f"  {'sweep: list namespaces':34} {type(exc).__name__}: {exc}")
        for _levels in _nss:
            _ns = ".".join(_levels) if isinstance(_levels, list) else str(_levels)
            for _kind, _lister, _dropper in (
                ("view", ic.list_views, ic.drop_view),
                ("table", ic.list_tables, ic.drop_table),
            ):
                try:
                    _r = _lister(FX.cat, _ns)
                    _ids = (_r.json().get("identifiers") or []) if _r.ok else []
                except Exception:
                    _ids = []
                for _id in _ids:
                    _nm = _id.get("name") if isinstance(_id, dict) else str(_id)
                    _try(f"sweep {_kind} {_ns}/{_nm}", lambda n=_ns, m=_nm, d=_dropper: d(FX.cat, n, m))
            _try(f"sweep namespace {_ns}", lambda n=_ns: ic.drop_namespace(FX.cat, n))

    if pc is not None:
        for _cr in (BINDING["catalogRoleName"], BINDING["doomed_catalog_role"],
                    BINDING["new_catalog_role"]):
            _try(f"delete catalog role {_cr}", lambda c=_cr: pc.delete_catalog_role(FX.cat, c))
        for _cat in (FX.cat, BINDING["doomed_catalog"], BINDING["new_catalog"]):
            _try(f"delete catalog {_cat}", lambda c=_cat: pc.delete_catalog(c))
        for _p in (BINDING["doomed_principal"], BINDING["new_principal"]):
            _try(f"delete principal {_p}", lambda p=_p: pc.delete_principal(p))
        for _pr in (BINDING["doomed_principal_role"], BINDING["new_principal_role"]):
            _try(f"delete principal role {_pr}",
                 lambda p=_pr: pc.delete_principal_role(p))
        for _name, _role in ((RUNNER_NAME, RUNNER_ROLE), (DENIED_NAME, DENIED_ROLE)):
            try:
                th.deprovision_run_principal(pc, _name, _role)
                out[f"deprovision {_name}"] = "ok"
            except Exception as exc:
                out[f"deprovision {_name}"] = f"{type(exc).__name__}: {exc}"
            if not quiet:
                print(f"  {'deprovision ' + _name:34} {out['deprovision ' + _name]}")
    return out


def _teardown_at_exit():
    if not _TORN["done"]:
        try:
            teardown_all(quiet=True)
        except Exception:
            pass


atexit.register(_teardown_at_exit)
print("  teardown registered: teardown_all() cleans up from anywhere, and atexit runs it on")
print("                       kernel shutdown if the run aborts before section 11")

## 3 -- The grid, printed BEFORE anything is driven

63 operations, 286 cells, from `log-coverage/spec/`. Every request is built once (dry run) before
any is issued, so a harness bug is found at call 0, not call 200.

In [ ]:
import collections

OPS = mx.load_spec(pathlib.Path.cwd() / "spec")
GRID = mx.cells(OPS)
BY_TARGET = collections.Counter(c.target for c in GRID)
BY_PHASE = collections.Counter(c.phase for c in GRID)

print(f"  {len(OPS)} operations from log-coverage/spec/  "
      f"({sum(1 for o in OPS if o.api == 'management')} management, "
      f"{sum(1 for o in OPS if o.api == 'catalog')} catalog)")
print(f"  {len(GRID)} cells   " + "  ".join(f"{k}={v}" for k, v in sorted(BY_TARGET.items(), key=str)))
print("  phases   " + "  ".join(f"{k}={v}" for k, v in sorted(BY_PHASE.items())))

print("\n  NOT REACHABLE on this build:")
for row in mx.ledger():
    mark = "settled" if row["state"] == "settled" else "PROBE  "
    print(f"    {str(row['status']):>4}  {mark}  {row['why']}")
print(f"  EXTRAS the spec never declares but this build returns: {mx.UNDECLARED_EXTRAS}")

# Every cell must build a request before ANY of them is issued. A missing payload
# or an unbindable path is a harness bug, and finding it at call 200 of 286 leaves
# a half-driven run on the cluster.
_dry = [mx.request_for(c, BINDING, TOKENS, "dry-run", ptu.REALM) for c in GRID]
print(f"\n  dry run  {len(_dry)} requests built, none raised")

ROWS, PHASES = [], {}


def run_phase(name, cells_, note=""):
    """Drive one phase immediately and record when it ran."""
    print(f"  phase {name}: {len(cells_)} cells {note}")
    t0 = time.time()
    rows = mx.drive(cells_, BINDING, TOKENS, ptu.POLARIS_URL, ptu.REALM, RUN,
                    seq_from=2000 + len(ROWS))
    t1 = time.time()
    PHASES[name] = {"start": _iso(t0), "end": _iso(t1),
                    "seconds": round(t1 - t0, 1), "cells": len(cells_)}
    ROWS.extend(rows)
    got = collections.Counter(r["verdict"] for r in rows)
    print(f"           {t1 - t0:.1f}s   {dict(got)}")
    return rows

## 4 -- Phase B: the 2xx sweep

Every operation at its own declared success code (`createCatalog` 201, `dropNamespace` 204).
`verdict` here is Polaris-side only: did the status match the target.

In [ ]:
PHASE_B = run_phase("B", [c for c in GRID if c.phase == "B" and c.target == 2])
_missed = [r for r in PHASE_B if r["verdict"] == mx.MISSED]
print(f"\n  {len(PHASE_B) - len(_missed)}/{len(PHASE_B)} covered")
for r in _missed[:20]:
    print(f"    MISS  {r['op_id']:34} got {r['status']}  {r['why']}")

## 5 -- Phase C: 401 / 403 / 404

In [ ]:
PHASE_C = run_phase("C", [c for c in GRID if c.phase == "C"])
_by = collections.Counter((r["target"], r["status"]) for r in PHASE_C)
print("\n  target -> actual:")
for (t, a), n in sorted(_by.items(), key=lambda kv: (str(kv[0][0]), str(kv[0][1]))):
    mark = "" if t == a else "   <-- MISS"
    print(f"    {t:>4} -> {str(a):>4}  x{n}{mark}")

## 6 -- Phase D: 409 / 400

409 from order on the creates and a stale `currentEntityVersion`; 400 from a body missing its
required fields, never from `{}`.

In [ ]:
PHASE_D = run_phase("D", [c for c in GRID if c.phase == "D"])
for target in (409, 400):
    rows = [r for r in PHASE_D if r["target"] == target]
    hit = sum(1 for r in rows if r["verdict"] == mx.COVERED)
    print(f"  {target}: {hit}/{len(rows)} covered")
    for r in rows:
        if r["verdict"] != mx.COVERED:
            print(f"    MISS  {r['op_id']:34} got {r['status']}")

## 7 -- Phase E: one table commit

Phase F's table is created first, so phase F has something to drop.

In [ ]:
from iceberg_rest import IcebergREST as _IR

run_ic = _IR(ptu.POLARIS_URL, ptu.REALM, token=TOKENS[mx.RUNNER])

# Phase F's table, created here so phase F has something to drop.
_drop_tbl = f"mx_{RUN}_droponly"
_r = run_ic.create_table(FX.cat, FX.ns, build_create_table_payload(_drop_tbl, SCHEMA))
print(f"  created {_drop_tbl}: {_r.status_code}")

_t0 = time.time()
_rid = mx.request_id(RUN, 3001, "gate2-the-commit")
th.tag_clients([run_ic], _rid)
_resp = run_ic.commit_table(FX.cat, FX.ns, FX.tbl,
                            [{"action": "set-properties", "updates": {"mx.gate2": RUN}}])
th.tag_clients([run_ic], None)
_t1 = time.time()
GATE2 = {
    "request_id": _rid, "status": _resp.status_code,
    "response_size": len(_resp.content or b""),
    "t0": _t0, "t1": _t1, "at": _iso(_t0),
    "table": FX.tbl,
}
PHASES["E"] = {"start": _iso(_t0), "end": _iso(_t1), "seconds": round(_t1 - _t0, 1), "cells": 1}
print(f"  commit {GATE2['status']}, {GATE2['response_size']} bytes (as the client received it), "
      f"at {GATE2['at']}")

## 8 -- Phase F: one table delete

In [ ]:
_t0 = time.time()
_rid = mx.request_id(RUN, 3002, "gate2-the-delete")
th.tag_clients([run_ic], _rid)
_resp = run_ic.drop_table(FX.cat, FX.ns, _drop_tbl)
th.tag_clients([run_ic], None)
_t1 = time.time()
GATE2_NEG = {
    "request_id": _rid, "status": _resp.status_code,
    "response_size": len(_resp.content or b""), "table": _drop_tbl,
    "t0": _t0, "t1": _t1, "at": _iso(_t0),
}
PHASES["F"] = {"start": _iso(_t0), "end": _iso(_t1), "seconds": round(_t1 - _t0, 1), "cells": 1}
print(f"  drop {GATE2_NEG['status']}, {GATE2_NEG['response_size']} bytes, at {GATE2_NEG['at']}")
if GATE2_NEG["response_size"] != 0:
    print("  !! the DELETE returned a non-empty body -- record it with the run.")

## 9 -- Phase G: three grants, and a denied read on the same role

`TABLE_READ_DATA`, `TABLE_WRITE_DATA`, `NAMESPACE_LIST` granted on one catalog role, then the
denied principal reads that role.

In [ ]:
GRANTS = ["TABLE_READ_DATA", "TABLE_WRITE_DATA", "NAMESPACE_LIST"]
_crole = BINDING["catalogRoleName"]

_t0 = time.time()

granted = []
for i, priv in enumerate(GRANTS):
    _rid = mx.request_id(RUN, 3100 + i, f"gate4-grant-{priv}")
    th.tag_clients([adm_pc], _rid)
    r = adm_pc.grant_privilege(FX.cat, _crole, priv)
    th.tag_clients([adm_pc], None)
    granted.append({"privilege": priv, "status": r.status_code, "request_id": _rid})
    print(f"    grant {priv:22} {r.status_code}")

# The denied principal reads the role it has no grants on. No successful request
# to this role from this identity -- which is the whole point.
denied_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=TOKENS[mx.DENIED])
_rid = mx.request_id(RUN, 3199, "gate4-denied-role-read")
th.tag_clients([denied_pc], _rid)
_dr = denied_pc.get_catalog_role(FX.cat, _crole)
th.tag_clients([denied_pc], None)
print(f"    denied read of the role: {_dr.status_code}")
_t1 = time.time()

GATE4 = {
    "role": _crole, "granted": granted, "t0": _t0, "t1": _t1,
    "expected_writes": sum(1 for g in granted if 200 <= (g["status"] or 0) < 300),
    "denied_status": _dr.status_code, "denied_request_id": _rid,
    "at": _iso(_t0),
}
PHASES["G"] = {"start": _iso(_t0), "end": _iso(_t1),
               "seconds": round(_t1 - _t0, 1), "cells": len(granted) + 1}
print(f"  successful grants on {_crole}: {GATE4['expected_writes']}  at {GATE4['at']}")

## 10 -- Phase H: the 500 ladder

`th.provokers_500`. Every rung answered a 4xx on 2026-09-07, so NOT PROVOKED is the expected
result on this build.

In [ ]:
PROVOKERS = th.provokers_500(
    adm_pc, run_ic, RUN, bucket=ptu.BUCKET, endpoint=ptu.MINIO_ENDPOINT,
    endpoint_internal=ptu.MINIO_ENDPOINT_INTERNAL,
    table_payload=lambda name: build_create_table_payload(name, SCHEMA), repeat=3)
print(f"  ladder: {[p.name for p in PROVOKERS]}")

_t0 = time.time()
_r500 = th.drive_500([adm_pc, run_ic], RUN, 3200, PROVOKERS,
                     on_rung=lambda e: print(f"    {e['rung']:22} calls={e['calls']} "
                                             f"statuses={e['statuses']} provoked={e['provoked']}"))
_t1 = time.time()

FIVE_HUNDRED = {
    "winner": _r500["winner"], "rows": _r500["rows"], "ladder": _r500["ladder"],
    "driven_500": len(th.classify_500s(_r500["rows"])["deliberate"]),
    "at": _iso(_t0),
}
PHASES["H"] = {"start": _iso(_t0), "end": _iso(_t1),
               "seconds": round(_t1 - _t0, 1), "cells": len(_r500["rows"])}

if FIVE_HUNDRED["winner"]:
    print(f"  PROVOKED by `{FIVE_HUNDRED['winner']}`: {FIVE_HUNDRED['driven_500']} deliberate 500(s)")
else:
    print("  NOT PROVOKED -- no rung returned >= 500. Expected on this build: "
          "IcebergExceptionMapper\n     catches storage failures and maps them to 4xx.")

## 11 -- Phase I: cleanup

`dropTable`, `dropView`, `dropNamespace`, `deleteCatalog` and `deletePrincipal` are five of the 63
operations, so cleanup is traffic too. `teardown_all` is idempotent.

In [ ]:
# The cleanup DELETEs are **part of the test**, not tidying: `dropTable`,
# `dropView`, `dropNamespace`, `deleteCatalog`, `deletePrincipal` are five of
# the 63 operations and rule 4 keeps every one of them. So they are run here,
# in the open, rather than left to the atexit hook.
#
# `teardown_all` was defined and registered in section 2, the moment the
# fixture existed, precisely so that an abort anywhere between there and here
# still cleans up. This cell is now the NORMAL path through it, not the only
# one. It is idempotent: if the hook already fired, it says so and does nothing.
CLEANUP = teardown_all()

## 12 -- The matrix, as Polaris answered it

One row per cell: `target` is what the cell asked for, `status` is what it got. A row where they
differ is MISSED with the actual code. Nothing here says anything about logs.

In [ ]:
import pandas as pd

M = pd.DataFrame(ROWS)
RESULTS = {(r["op_id"], r["target"]): {"verdict": r["verdict"], "actual": r["status"],
                                       "why": r["why"]} for r in ROWS}
COVERAGE = mx.coverage_by_operation(OPS, RESULTS)

print("  by target:")
for target in sorted({r["target"] for r in ROWS}, key=str):
    rows = [r for r in ROWS if r["target"] == target]
    cov = sum(1 for r in rows if r["verdict"] == mx.COVERED)
    err = sum(1 for r in rows if r["verdict"] == mx.ERROR)
    print(f"    {str(target):>4}  {cov:>3}/{len(rows):<3} covered"
          + (f"   {err} transport error(s)" if err else ""))

_full = [c for c in COVERAGE if not c["missed"] and not c["not_driven"]]
print(f"\n  operations with EVERY planned cell covered: {len(_full)}/{len(OPS)}")

_holes = [c for c in COVERAGE if c["missed"]]
print(f"  operations with a missed cell: {len(_holes)}")
for c in _holes[:25]:
    got = {t: RESULTS[(c['op_id'], t)]['actual'] for t in c['missed']}
    print(f"    {c['op_id']:34} missed {c['missed']}  got {got}")

pd.set_option("display.width", 200, "display.max_rows", 400, "display.max_colwidth", 44)
print("\n" + M[["op_id", "method", "target", "status", "verdict", "principal", "phase",
                "elapsed_ms"]].to_string(index=False))

## 13 -- The run, for the verifier

Everything `local-k8s` needs to find this run: the run id, when each phase ran, the request ids of the
single-purpose phases, and every >= 500 Polaris returned. No verdicts.

In [ ]:
print(f"  run {RUN}  env={ptu.ENV}  polaris={ptu.POLARIS_URL}  realm={ptu.REALM}")
print(f"  started {STARTED.isoformat()}  finished {datetime.now(timezone.utc).isoformat()}")
print(f"  driven as {DRIVE_PRINCIPAL} (elected: {IDENTITY})")

print("\n  phases:")
for name, w in sorted(PHASES.items()):
    print(f"    {name}  {w['start']} .. {w['end']}  {w['seconds']:>6.1f}s  {w['cells']:>3} calls")

print("\n  single-purpose request ids:")
print(f"    E commit   {GATE2['request_id']}  status={GATE2['status']}  "
      f"bytes={GATE2['response_size']}  table={GATE2['table']}  at={GATE2['at']}")
print(f"    F delete   {GATE2_NEG['request_id']}  status={GATE2_NEG['status']}  "
      f"bytes={GATE2_NEG['response_size']}  table={GATE2_NEG['table']}  at={GATE2_NEG['at']}")
for g in GATE4["granted"]:
    print(f"    G grant    {g['request_id']}  status={g['status']}  {g['privilege']}  "
          f"role={GATE4['role']}")
print(f"    G denied   {GATE4['denied_request_id']}  status={GATE4['denied_status']}  "
      f"role={GATE4['role']}  at={GATE4['at']}")

ALL_500 = [r for r in ROWS if (r["status"] or 0) >= 500]
print(f"\n  >= 500 in the grid: {len(ALL_500)}   (ladder: "
      f"{FIVE_HUNDRED['winner'] or 'NOT PROVOKED'}, {FIVE_HUNDRED['driven_500']} deliberate)")
for r in ALL_500:
    print(f"    {r['op_id']:24} target {r['target']:>4} -> {r['status']}  "
          f"as {r['principal']}  phase {r['phase']}  {r['request_id']}")

## 14 -- Teardown

Close the HTTP sessions and drop the frame so a re-run starts clean.

In [ ]:
import gc

for _c in (adm_pc, adm_ic, run_ic, denied_pc):
    try:
        _c.session.close()
    except Exception:
        pass

del M
gc.collect()
print("  sessions closed, frame released")
print(f"\n  run {RUN} complete -- {len(ROWS)} grid calls, phases {sorted(PHASES)}")